# Embed Additional Data via Lambda Labs

Uploads normalized tar archives to Lambda, polls for results, downloads embeddings.

**Prerequisites:**
1. Run `Prepare_Additional_For_Embedding.ipynb` first to create tar archives in `D:\temp\`
2. SSH into Lambda and start `run_embeddings_fast.py`

**Flow:** Upload tar → Lambda embeds → Download parquet → Verify → Move to collection folder

**Output:** `D:\hist_LLM\additional_data\embeddings\{collection}\embeddings_{year}.parquet`

In [ ]:
import subprocess
import os
import time
import re
from pathlib import Path

# --- CONFIG ---
TAR_DIR = Path(r"D:\temp")
RESULT_BASE = Path(r"D:\hist_LLM\additional_data\embeddings")  # {collection}/embeddings_{year}.parquet
RESULT_BASE.mkdir(parents=True, exist_ok=True)

# Lambda connection
REMOTE_USER_IP = "ubuntu@209.20.159.29"
REMOTE_INBOX_DIR = "/lambda/nfs/embedding/input_data/"
REMOTE_OUTBOX_PATH = "/lambda/nfs/embedding/output_data/"
SSH_KEY = r"C:\Users\danielyoon\Documents\lambda_nanochat_openssh"

# SSH options to prevent hanging on host key prompts or stale connections
SSH_OPTS = ["-o", "StrictHostKeyChecking=no", "-o", "ConnectTimeout=30"]

# Source code → collection name mapping
CODE_TO_COLLECTION = {
    "nyt": "nyt",
    "economist": "economist",
    "ft": "ft",
    "newswire": "newswire",
}

print(f"Tar directory: {TAR_DIR}")
print(f"Result directory: {RESULT_BASE}")
print(f"Lambda: {REMOTE_USER_IP}")
print(f"Remote inbox: {REMOTE_INBOX_DIR}")

# Quick connectivity test
print("\nTesting SSH connection...")
test = subprocess.run(
    ["ssh", "-i", SSH_KEY] + SSH_OPTS + [REMOTE_USER_IP, "echo connected"],
    capture_output=True, text=True, timeout=30
)
if "connected" in test.stdout:
    print("  [OK] SSH connection works!")
else:
    print(f"  [FAIL] SSH error: {test.stderr.strip()}")
    print("  Check REMOTE_USER_IP and SSH_KEY above.")

## Helper Functions

In [19]:
def parse_tar_name(tar_path):
    """Parse tar filename to extract source code and year.
    
    year_nyt1851.tar → ('nyt', 1851)
    year_economist1900.tar → ('economist', 1900)
    year_ft1888.tar → ('ft', 1888)
    year_newswire1878.tar → ('newswire', 1878)
    """
    stem = tar_path.stem  # year_nyt1851
    after_year = stem[len("year_"):]  # nyt1851
    # Find where digits start
    match = re.match(r'([a-z]+)(\d+)', after_year)
    if match:
        return match.group(1), int(match.group(2))
    return None, None


def verify_parquet_file(filepath):
    """Verify parquet file has PAR1 magic bytes at start and end."""
    if not filepath.exists():
        return False
    try:
        with open(filepath, 'rb') as f:
            if f.read(4) != b'PAR1':
                return False
            f.seek(-4, 2)
            if f.read(4) != b'PAR1':
                print(f"    [WARN] Missing end magic bytes (truncated)")
                return False
        return True
    except Exception as e:
        print(f"    [WARN] Verification error: {e}")
        return False


def get_result_path(source_code, year):
    """Get the final result path for a given source and year."""
    collection = CODE_TO_COLLECTION.get(source_code, source_code)
    result_dir = RESULT_BASE / collection
    result_dir.mkdir(parents=True, exist_ok=True)
    return result_dir / f"embeddings_{year}.parquet"


def is_already_done(source_code, year):
    """Check if embedding already exists for this source/year."""
    return get_result_path(source_code, year).exists()


print("Helpers loaded.")

Helpers loaded.


## Check What's Ready

In [20]:
# Find all tar files to process
all_tars = sorted(TAR_DIR.glob("year_*.tar"))

# Filter out English tars (year_NNNN.tar where it's just a number)
additional_tars = []
for t in all_tars:
    source_code, year = parse_tar_name(t)
    if source_code is not None:
        additional_tars.append(t)

print(f"Total additional data tar files: {len(additional_tars)}")

# Check which are already done
todo = []
done = []
for t in additional_tars:
    source_code, year = parse_tar_name(t)
    if is_already_done(source_code, year):
        done.append(t)
    else:
        todo.append(t)

print(f"Already done: {len(done)}")
print(f"To process: {len(todo)}")

if todo:
    # Show breakdown by source
    by_source = {}
    for t in todo:
        sc, _ = parse_tar_name(t)
        by_source[sc] = by_source.get(sc, 0) + 1
    print("\nBreakdown:")
    for sc, count in sorted(by_source.items()):
        print(f"  {sc}: {count} files")

Total additional data tar files: 300
Already done: 2
To process: 298

Breakdown:
  ft: 32 files
  newswire: 99 files
  nyt: 167 files


## Run Embedding Workflow

**Before running this cell**, make sure:
1. Lambda server is up and accessible
2. `run_embeddings_fast.py` is running on Lambda
3. Directories exist on Lambda: `input_data/`, `output_data/`, `temp_extract/`

In [21]:
# Optional: filter to specific source(s)
# Set to None to process all, or e.g. ["nyt"] to only process NYT
FILTER_SOURCES = None  # e.g., ["nyt"], ["ft", "economist"], or None for all

if FILTER_SOURCES:
    filtered_todo = [t for t in todo if parse_tar_name(t)[0] in FILTER_SOURCES]
    print(f"Filtered to {len(filtered_todo)} files (sources: {FILTER_SOURCES})")
else:
    filtered_todo = todo
    print(f"Processing all {len(filtered_todo)} files")

Processing all 298 files


In [25]:
for tar_path in filtered_todo:
    source_code, year = parse_tar_name(tar_path)
    
    # Double-check not already done
    if is_already_done(source_code, year):
        print(f"[SKIP] {source_code} {year}: already done")
        continue
    
    archive_name = tar_path.name  # year_nyt1851.tar
    # Lambda script extracts stem.split("_")[1] as the "year" label for output
    # So year_nyt1851.tar → stem = year_nyt1851 → split("_")[1] = nyt1851
    # → output: embeddings_nyt1851.parquet
    lambda_output_name = f"embeddings_{source_code}{year}.parquet"
    
    print(f"\n{'='*50}")
    print(f"Processing: {source_code} {year}")
    print(f"{'='*50}")
    
    # --- Step 1: UPLOAD ---
    temp_remote_path = f"{REMOTE_INBOX_DIR}{archive_name}.tmp"
    final_remote_path = f"{REMOTE_INBOX_DIR}{archive_name}"
    
    tar_mb = tar_path.stat().st_size / 1024 / 1024
    print(f">>> Uploading {archive_name} ({tar_mb:.1f} MB)...")
    result = subprocess.run(
        ["scp", "-i", SSH_KEY] + SSH_OPTS + [str(tar_path), f"{REMOTE_USER_IP}:{temp_remote_path}"],
        capture_output=True, text=True, timeout=3600  # 10 min — FT tars can be 100+ MB
    )
    if result.returncode != 0:
        print(f"    [ERROR] SCP failed: {result.stderr}")
        continue
    
    # Atomic move to trigger Lambda processing
    result = subprocess.run(
        ["ssh", "-i", SSH_KEY] + SSH_OPTS + [REMOTE_USER_IP, f"mv {temp_remote_path} {final_remote_path}"],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode != 0:
        print(f"    [ERROR] mv failed: {result.stderr}")
        continue
    
    # --- Step 2: POLL FOR RESULTS ---
    remote_result_file = f"{REMOTE_OUTBOX_PATH}{lambda_output_name}"
    print(f">>> Waiting for Lambda to produce {lambda_output_name}...")
    
    while True:
        check_cmd = ["ssh", "-i", SSH_KEY] + SSH_OPTS + [REMOTE_USER_IP, f"test -f {remote_result_file} && echo 'exists'"]
        result = subprocess.run(check_cmd, capture_output=True, text=True, timeout=180)
        if "exists" in result.stdout:
            print(f">>> Results ready!")
            break
        time.sleep(30)
    
    # --- Step 3: DOWNLOAD ---
    # Download to a temp location first, then move to final path
    final_path = get_result_path(source_code, year)
    temp_download = final_path.parent / f"_temp_{final_path.name}"
    
    print(f">>> Downloading to {final_path}...")
    result = subprocess.run(
        ["scp", "-i", SSH_KEY] + SSH_OPTS + [f"{REMOTE_USER_IP}:{remote_result_file}", str(temp_download)],
        capture_output=True, text=True, timeout=3600  # 30 min — embedding parquets can be 300+ MB
    )
    if result.returncode != 0:
        print(f"    [ERROR] Download failed: {result.stderr}")
        continue
    
    # --- Step 4: VERIFY & FINALIZE ---
    if verify_parquet_file(temp_download):
        # Move temp to final
        temp_download.rename(final_path)
        size_mb = final_path.stat().st_size / 1024 / 1024
        print(f"    [OK] Verified ({size_mb:.1f} MB). Deleting remote...")
        subprocess.run(
            ["ssh", "-i", SSH_KEY] + SSH_OPTS + [REMOTE_USER_IP, f"rm {remote_result_file}"],
            capture_output=True, text=True, timeout=30
        )
        # Delete the local tar since we're done with it
        tar_path.unlink()
        print(f">>> {source_code} {year} complete!")
    else:
        print(f"    [FAIL] Corrupted! Keeping remote, deleting local temp.")
        if temp_download.exists():
            temp_download.unlink()
        print(f">>> {source_code} {year} FAILED - will retry on next run.")

print("\n" + "="*50)
print("WORKFLOW COMPLETE")
print("="*50)

[SKIP] ft 1975: already done
[SKIP] ft 1976: already done
[SKIP] ft 1977: already done
[SKIP] ft 1978: already done
[SKIP] ft 1979: already done
[SKIP] ft 1980: already done
[SKIP] ft 1981: already done
[SKIP] ft 1982: already done
[SKIP] ft 1983: already done
[SKIP] ft 1984: already done
[SKIP] ft 1985: already done
[SKIP] ft 1986: already done
[SKIP] ft 1987: already done
[SKIP] ft 1988: already done
[SKIP] ft 1989: already done
[SKIP] ft 1990: already done
[SKIP] ft 1991: already done
[SKIP] ft 1992: already done
[SKIP] ft 1993: already done
[SKIP] ft 1994: already done
[SKIP] ft 1995: already done
[SKIP] ft 1996: already done
[SKIP] ft 1997: already done
[SKIP] ft 1998: already done
[SKIP] ft 1999: already done
[SKIP] ft 2000: already done
[SKIP] ft 2001: already done
[SKIP] ft 2002: already done
[SKIP] ft 2003: already done
[SKIP] ft 2004: already done
[SKIP] ft 2005: already done
[SKIP] ft 2006: already done
[SKIP] newswire 1878: already done
[SKIP] newswire 1879: already done
[S

## Status Check

In [ ]:
print("Embedding results by collection:\n")
for collection in ["nyt", "economist", "ft", "newswire"]:
    coll_dir = RESULT_BASE / collection
    if coll_dir.exists():
        files = sorted(coll_dir.glob("embeddings_*.parquet"))
        total_mb = sum(f.stat().st_size / 1024 / 1024 for f in files)
        if files:
            years = [int(f.stem.split("_")[1]) for f in files]
            print(f"{collection}: {len(files)} files, {total_mb:.1f} MB")
            print(f"  Years: {min(years)}-{max(years)}")
        else:
            print(f"{collection}: no files yet")
    else:
        print(f"{collection}: directory not created yet")

# Check remaining tars
remaining_tars = [t for t in TAR_DIR.glob("year_*.tar") if parse_tar_name(t)[0] is not None]
if remaining_tars:
    print(f"\nRemaining tar files to process: {len(remaining_tars)}")

## Sanity Check
Load one embedding file and verify shape.

In [ ]:
import pandas as pd
import numpy as np

# Find first available embedding file
sample_file = None
for collection in ["nyt", "economist", "ft", "newswire"]:
    coll_dir = RESULT_BASE / collection
    if coll_dir.exists():
        files = list(coll_dir.glob("embeddings_*.parquet"))
        if files:
            sample_file = files[0]
            break

if sample_file:
    df = pd.read_parquet(sample_file)
    print(f"File: {sample_file}")
    print(f"Columns: {list(df.columns)}")
    print(f"Rows: {len(df):,}")
    
    # Check embedding dimension
    emb = np.array(df.iloc[0]['embedding'])
    print(f"Embedding dimension: {emb.shape[0]}")
    print(f"Embedding dtype: {emb.dtype}")
    print(f"\nFirst row identifier: {df.iloc[0]['original_index']}")
else:
    print("No embedding files found yet. Run the workflow first.")